# Taller
Este cuaderno es para trabajar sobre su propio proyecto. No hay que ejecutarlo completo: cada persona sigue la hoja de ruta y usa solo la sección que corresponde a sus datos.

La meta de hoy es salir con:

1. Una tarea bien definida.
2. Un modelo escogido **y justificado**.
3. Una prueba con un archivo, y luego con varios.
4. Una revisión honesta de los resultados.
5. El esquema de la presentación del lunes 23.

**Hoja de ruta**

| Paso | Qué hacer | Tiempo aproximado |
|---|---|---|
| 1 | Definir la tarea | 5 min |
| 2 | Escoger el modelo con las 6 preguntas | 5 min |
| 3 | Cargar sus datos | 5 min |
| 4 | Probar con **un** archivo | 10 min |
| 5 | Procesar **varios** archivos | 10 min |
| 6 | Revisar los resultados | 5 min |
| 7 | Armar el esquema de la presentación | al final y en casa |



## Paso 1 · Definir la tarea

Una buena tarea cabe en una frase: **"Quiero usar [modelo] para [hacer algo] con [mis datos] y así responder [pregunta]"**.

Por ejemplo: *"Quiero transcribir 8 entrevistas para buscar en qué momentos hablan de seguridad."*

Esta tabla les ayuda a ubicar su tarea:

| Mis datos son... | Tarea | Lo vimos en | Ruta del taller |
|---|---|---|---|
| Audios, entrevistas | Transcripción  | **A** |
| Fotos | Detectar y contar objetos | **B1** |
| Fotos | Clasificar con mis propias categorías | **B2** |
| Documentos escaneados, PDF | Leer el texto (OCR)  | **C** |
| Textos (respuestas abiertas, mensajes, transcripciones) | Sentimiento, temas | **D** |

Si su proyecto combina varias rutas (por ejemplo, transcribir audios y después analizar el texto), hagan primero una y después la otra.



## Paso 2 · Escoger el modelo

Busquen en [huggingface.co/models](https://huggingface.co/models) filtrando por la tarea y el idioma. También pueden buscar desde aquí: cambien el valor de `tarea` y ejecuten.

Nombres de tareas útiles:
- `"automatic-speech-recognition"`: transcripción
- `"object-detection"`: detectar objetos
- `"zero-shot-image-classification"`: clasificar imágenes con etiquetas propias
- `"image-to-text"`: describir imágenes
- `"text-classification"`: clasificar textos (sentimiento, temas)

In [ ]:
from huggingface_hub import list_models

tarea = "automatic-speech-recognition"

modelos = list_models(task=tarea, sort="downloads", limit=15)

for modelo in modelos:
    print(modelo.id, modelo.downloads)

Escojan un modelo y respondan las 6 preguntas.

- **Modelo escogido:**
- **1. ¿Hace la tarea que necesitamos?**
- **2. ¿Funciona en nuestro idioma? ¿Con qué calidad?**
- **3. ¿Qué tamaño tiene? ¿Corre en Colab?**
- **4. ¿Qué licencia tiene?**
- **5. ¿Con qué datos se entrenó? ¿Qué limitaciones declara?**
- **6. ¿Tiene documentación y ejemplo de uso?**
- **¿Qué otro modelo consideramos y por qué no lo escogimos?**

## Paso 3 · Cargar sus datos

Escojan una de las dos opciones.

**Opción A: Google Drive.** Es la mejor si tienen muchos archivos, porque no se borran cuando Colab se reinicia. Pongan sus archivos en una carpeta de su Drive y ejecuten:

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
carpeta = "/content/drive/MyDrive/nombre_de_su_carpeta"

Cambien `nombre_de_su_carpeta` por el nombre real de su carpeta. Si la carpeta está dentro de otra, escriban el camino completo, por ejemplo `"/content/drive/MyDrive/proyecto/audios"`.

**Opción B: subir archivos.** Sirve para pocos archivos. Se borran cuando Colab se reinicia.

In [ ]:
import os
from google.colab import files

os.makedirs("mis_datos", exist_ok=True)

archivos_subidos = files.upload()

for nombre in archivos_subidos:
    os.rename(nombre, "mis_datos/" + nombre)

carpeta = "mis_datos"

In [ ]:
from glob import glob

archivos = sorted(glob(carpeta + "/*"))

print(len(archivos), "archivos")
print(archivos[:5])

Si aparece `0 archivos`, revisen el nombre de la carpeta en `carpeta`. Las mayúsculas, las tildes y los espacios cuentan.

Si en la carpeta hay archivos de distintos tipos, pueden quedarse solo con uno. Por ejemplo, solo los `.mp3`:

```python
archivos = sorted(glob(carpeta + "/*.mp3"))
```

---
# Pasos 4 y 5 · Probar y procesar

Vayan **solo** a la ruta de su proyecto. En cada ruta:

- **Paso 4:** prueben con **un** archivo y miren el resultado con calma.
- **Paso 5:** cuando el paso 4 funcione, procesen **varios** archivos con un `for`. Empiecen con 3 (`archivos[:3]`) y, si todo va bien, usen todos.

En todas las rutas el resultado final es una tabla llamada `tabla`, que se usa en el paso 6.

---
## Ruta A · Audio: transcribir entrevistas

### Paso 4: un archivo

In [ ]:
import pandas as pd
import torch
from IPython.display import Audio
from transformers import pipeline

if torch.cuda.is_available():
    dispositivo = "cuda"
else:
    dispositivo = "cpu"

transcriptor = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    device=dispositivo
)

Si escogieron otro modelo en el paso 2, cambien el nombre en `model=`.

In [ ]:
ruta = archivos[0]

Audio(ruta)

In [ ]:
resultado = transcriptor(
    ruta,
    generate_kwargs={"language": "spanish", "task": "transcribe"},
    return_timestamps=True
)

print(resultado["text"])

**Para revisar:** escuchen el audio mientras leen. ¿Qué errores hay? ¿En qué partes del audio aparecen (ruido, varias personas hablando, nombres)?

### Paso 5: varios archivos

Guardamos una fila por archivo. Si son audios largos, cada uno puede tardar varios minutos.

In [ ]:
transcripciones = []

for ruta in archivos[:3]:
    print("Transcribiendo", ruta)
    resultado = transcriptor(
        ruta,
        generate_kwargs={"language": "spanish", "task": "transcribe"},
        return_timestamps=True
    )
    ficha = {
        "archivo": ruta.split("/")[-1],
        "texto": resultado["text"],
        "palabras": len(resultado["text"].split()),
    }
    transcripciones.append(ficha)

tabla = pd.DataFrame(transcripciones)
tabla

---
## Ruta B1 · Imágenes: detectar y contar objetos

### Paso 4: un archivo

In [ ]:
!pip install -q ultralytics

In [ ]:
import pandas as pd
from PIL import Image
from ultralytics import YOLO

modelo = YOLO("yolo26n.pt")

In [ ]:
ruta = archivos[0]

resultado = modelo(ruta)[0]

resultado.save("foto_detectada.jpg")
Image.open("foto_detectada.jpg")

In [ ]:
for caja in resultado.boxes:
    nombre = resultado.names[int(caja.cls)]
    confianza = float(caja.conf)
    print(nombre, round(confianza, 2))

In [ ]:
detecciones = []

for ruta in archivos[:3]:
    resultado = modelo(ruta, verbose=False)[0]
    for caja in resultado.boxes:
        ficha = {
            "archivo": ruta.split("/")[-1],
            "objeto": resultado.names[int(caja.cls)],
            "confianza": round(float(caja.conf), 2),
        }
        detecciones.append(ficha)

tabla = pd.DataFrame(detecciones)
tabla

Cuántos objetos de cada tipo hay en cada foto:

In [ ]:
pd.crosstab(tabla["archivo"], tabla["objeto"])

---
## Ruta B2 · Imágenes: clasificar con sus propias categorías

Aquí el modelo (CLIP) escoge, entre **las etiquetas que ustedes escriben**, cuál describe mejor cada foto. Recuerden lo que vimos en la clase 6: suele funcionar mejor con etiquetas en inglés.

### Paso 4: un archivo

In [ ]:
import pandas as pd
from PIL import Image
from transformers import pipeline

clasificador = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

In [ ]:
etiquetas = ["a street market", "an empty street", "a park", "a construction site"]

Cambien la lista `etiquetas` por las categorías de su proyecto.

In [ ]:
ruta = archivos[0]
imagen = Image.open(ruta).convert("RGB")

respuesta = clasificador(imagen, candidate_labels=etiquetas)

for opcion in respuesta:
    print(opcion["label"], round(opcion["score"], 2))

imagen

**Para revisar:** ¿la etiqueta con el puntaje más alto es la correcta? Si dos etiquetas tienen puntajes muy parecidos, el modelo está dudando. ¿Las categorías son claras y no se mezclan entre sí?

### Paso 5: varios archivos

In [ ]:
clasificaciones = []

for ruta in archivos[:3]:
    imagen = Image.open(ruta).convert("RGB")
    respuesta = clasificador(imagen, candidate_labels=etiquetas)
    mejor = respuesta[0]
    ficha = {
        "archivo": ruta.split("/")[-1],
        "etiqueta": mejor["label"],
        "confianza": round(mejor["score"], 2),
    }
    clasificaciones.append(ficha)

tabla = pd.DataFrame(clasificaciones)
tabla

In [ ]:
tabla["etiqueta"].value_counts()

---
## Ruta C · Documentos: leer el texto con OCR

### Paso 4: un archivo

In [ ]:
!pip install -q easyocr pymupdf

In [ ]:
import easyocr
import numpy as np
import pandas as pd
import pymupdf
from PIL import Image

lector = easyocr.Reader(["es"])

**Si sus documentos son fotos o escaneos** (`.jpg`, `.png`):

In [ ]:
ruta = archivos[0]

resultados = lector.readtext(ruta)

for caja, texto_leido, confianza in resultados:
    print(round(confianza, 2), texto_leido)

**Si sus documentos son PDF**, primero convertimos una página en imagen, igual que en la clase 7:

In [ ]:
documento = pymupdf.open(archivos[0])

pagina = documento[0]
foto = pagina.get_pixmap(dpi=150)
imagen = Image.frombytes("RGB", [foto.width, foto.height], foto.samples)

resultados = lector.readtext(np.array(imagen))

for caja, texto_leido, confianza in resultados:
    print(round(confianza, 2), texto_leido)

In [ ]:
lecturas = []

for ruta in archivos[:3]:
    resultados = lector.readtext(ruta)
    for caja, texto_leido, confianza in resultados:
        ficha = {
            "archivo": ruta.split("/")[-1],
            "texto": texto_leido,
            "confianza": round(confianza, 2),
        }
        lecturas.append(ficha)

tabla = pd.DataFrame(lecturas)
tabla

Para un PDF con varias páginas (una fila por cada texto leído, indicando la página):

In [ ]:
lecturas = []

for numero, pagina in enumerate(documento):
    foto = pagina.get_pixmap(dpi=150)
    imagen_pagina = Image.frombytes("RGB", [foto.width, foto.height], foto.samples)
    resultados = lector.readtext(np.array(imagen_pagina))
    for caja, texto_leido, confianza in resultados:
        ficha = {
            "pagina": numero + 1,
            "texto": texto_leido,
            "confianza": round(confianza, 2),
        }
        lecturas.append(ficha)

tabla = pd.DataFrame(lecturas)
tabla

---
## Ruta D · Textos: sentimiento y temas

Sus textos pueden venir de un archivo `.csv` (una fila por respuesta o mensaje) o de la tabla de la **Ruta A**.

### Paso 4: un texto

In [ ]:
!pip install -q pysentimiento

In [ ]:
import pandas as pd
from pysentimiento import create_analyzer

analizador = create_analyzer(task="sentiment", lang="es")

**Si sus textos están en un `.csv`**, cárguenlo y miren cómo se llama la columna con el texto:

In [ ]:
datos = pd.read_csv(archivos[0])
datos.head()

In [ ]:
columna = "texto"

Cambien `"texto"` por el nombre real de su columna. **Si vienen de la Ruta A**, usen `datos = tabla` y dejen `columna = "texto"`.

In [ ]:
primer_texto = datos[columna][0]

prediccion = analizador.predict(primer_texto)

print(primer_texto)
print(prediccion.output, prediccion.probas)

In [ ]:
sentimientos = []

for texto_actual in datos[columna][:10]:
    prediccion = analizador.predict(texto_actual)
    sentimientos.append(prediccion.output)

tabla = datos[:10].copy()
tabla["sentimiento"] = sentimientos
tabla

In [ ]:
tabla["sentimiento"].value_counts()

**Temas con palabras clave** (sin modelo, como en la clase 9). Cambien el tema y las palabras:

In [ ]:
tabla["seguridad"] = tabla[columna].str.contains("robo|miedo|policía|inseguro", case=False)

tabla["seguridad"].sum()

---
## Paso 6 · Revisar los resultados

Un modelo siempre se equivoca en algo. Lo importante es **saber en qué** y **cuánto**. Este paso es igual para todas las rutas.

**1. Revisión a mano.** Tomen 5 filas al azar y compárenlas con los datos reales (escuchen el audio, miren la foto, lean el documento):

In [ ]:
tabla.sample(5)

Anoten en esta celda (doble clic para editar):

| Archivo | Qué dijo el modelo | Qué es en realidad | ¿Correcto? |
|---|---|---|---|
| | | | |
| | | | |
| | | | |
| | | | |
| | | | |

**De 5, el modelo acertó en:**

**2. Los casos dudosos.** Si su tabla tiene una columna `confianza` (rutas B1, B2 y C), miren los resultados con confianza baja:

In [ ]:
tabla[tabla["confianza"] < 0.5]

**3. Guardar.** Guarden la tabla para usarla en la presentación:

In [ ]:
from google.colab import files

tabla.to_csv("resultados_proyecto.csv", index=False)

files.download("resultados_proyecto.csv")

Si usaron la Opción A (Drive), también pueden guardarla directamente en su carpeta:

```python
tabla.to_csv(carpeta + "/resultados_proyecto.csv", index=False)
```



## Paso 7 · Esquema de la presentación (lunes 23)

Completen este esquema. Cada punto puede ser una diapositiva.

1. **Pregunta y contexto:** ¿qué queríamos saber y por qué importa?
2. **Datos:** qué son, cuántos, de dónde vienen y cómo se obtuvo el consentimiento, si aplica.
3. **Modelo y justificación:** qué modelo usamos y por qué. Qué otro consideramos.
4. **Cómo lo hicimos:** los pasos, en pocas palabras. No hace falta mostrar código.
5. **Un resultado:** un ejemplo concreto (una transcripción, una foto con detecciones, una tabla, un gráfico).
6. **Qué tan bien funcionó:** la revisión a mano del paso 6. Dónde se equivoca.
7. **Limitaciones y ética:** a quién deja fuera, dónde viajan los datos, qué se pierde.
8. **Siguiente paso:** qué harían con más tiempo o más datos.



---
## Si algo falla

| Mensaje o síntoma | Qué suele significar | Qué probar |
|---|---|---|
| `FileNotFoundError` o `0 archivos` | La ruta o el nombre del archivo están mal | Revisen `carpeta` y ejecuten `print(archivos)` |
| `CUDA out of memory` | El modelo no cabe en la GPU | Usen un modelo más pequeño, o *Entorno de ejecución* → *Reiniciar sesión* |
| Tarda muchísimo | No hay GPU, o hay muchos archivos o muy largos | Revisen que la GPU esté activa y prueben con `archivos[:3]` |
| `NameError: name '...' is not defined` | Falta ejecutar una celda anterior | Ejecuten las celdas de su ruta en orden, desde el principio |
| Fotos de iPhone (`.heic`) que no abren | Formato no compatible | Conviértanlas a `.jpg` antes de subirlas |
| Audio de WhatsApp (`.opus`, `.ogg`) | Suele funcionar; si no, hay que convertirlo | Conviértanlo a `.mp3` antes de subirlo |
| Se borraron los archivos | Colab se reinició (Opción B) | Usen Google Drive (Opción A) |
| `KeyError` con el nombre de una columna | La columna se llama distinto | Revisen con `datos.columns` |